In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
#
import numpy as np  # linear algebra
import os

# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
    import pandas
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import time

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

for dirname, _, filenames in os.walk("../input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Importing libs
import numpy as np

# STEFANOS: Disable unneeded modules
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import KMeans, AffinityPropagation
# import matplotlib.pyplot as plt
# import seaborn as sns
# %matplotlib inline
import warnings

warnings.filterwarnings("ignore")
# import plotly as py
# import plotly.graph_objs as go
import os

# py.offline.init_notebook_mode(connected = True)
# print(os.listdir("../input"))
import datetime as dt
# STEFANOS: Disable unneeded modules
# import missingno as msno
# plt.rcParams['figure.dpi'] = 140

In [ ]:
### cell 0 ###

df = pd.read_csv(
    "/home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/input/netflix-shows/netflix_titles.csv"
)
factor = 500
df = pd.concat([df] * factor, ignore_index=True)
df.head(3)

In [ ]:
### cell 1 ###

# Missing data

for i in df.columns:
    null_rate = df[i].isna().sum() / len(df) * 100
    if null_rate > 0:
        print("{} null rate: {}%".format(i, round(null_rate, 2)))

In [ ]:
### cell 2 ###

# Compute mode for country once
mode_country = df["country"].mode().iat[0]

# Fill missing values in one pass
fill_map = {"country": mode_country, "cast": "No Data", "director": "No Data"}
df.fillna(fill_map, inplace=True)

# Drop any remaining rows with missing values
df.dropna(inplace=True)

In [ ]:
### cell 3 ###

df.shape[0] - df.count()

In [ ]:
### cell 4 ###

df.info()

In [ ]:
### cell 5 ###

df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")


df["month_added"] = df["date_added"].dt.month
df["index"] = df["date_added"].dt.month_name()
df["year_added"] = df["date_added"].dt.year

df.head(3)

In [ ]:
### cell 6 ###

# For viz: Ratio of Movies & TV shows
x = df.groupby("type").size()
mf_ratio = x.div(len(df)).round(2).to_frame().T

In [ ]:
### cell 7 ###

for i_1 in mf_ratio.index:
    _ = int(mf_ratio["Movie"][i_1] * 100)
    _ = mf_ratio["Movie"][i_1] / 2
    _ = mf_ratio["Movie"][i_1] / 2

for i_2 in mf_ratio.index:
    _ = int(mf_ratio["TV Show"][i_2] * 100)
    _ = mf_ratio["Movie"][i_2] + mf_ratio["TV Show"][i_2] / 2
    _ = mf_ratio["Movie"][i_2] + mf_ratio["TV Show"][i_2] / 2

In [ ]:
### cell 8 ###

# Quick feature engineering

# Helper column for various plots
df["count"] = 1

# Many productions have several countries listed - this will skew our results , we'll grab the first one mentioned

# Lets retrieve just the first country
df["first_country"] = df["country"].apply(lambda x: x.split(",")[0])
df["first_country"].head()

# Rating ages from this notebook: https://www.kaggle.com/andreshg/eda-beginner-to-expert-plotly (thank you!)

ratings_ages = {
    "TV-PG": "Older Kids",
    "TV-MA": "Adults",
    "TV-Y7-FV": "Older Kids",
    "TV-Y7": "Older Kids",
    "TV-14": "Teens",
    "R": "Adults",
    "TV-Y": "Kids",
    "NR": "Adults",
    "PG-13": "Teens",
    "TV-G": "Kids",
    "PG": "Older Kids",
    "G": "Kids",
    "UR": "Adults",
    "NC-17": "Adults",
}

df["target_ages"] = df["rating"].replace(ratings_ages)
df["target_ages"].unique()

# Genre

df["genre"] = df["listed_in"].apply(
    lambda x: x.replace(" ,", ",").replace(", ", ",").split(",")
)

# Reducing name length

df["first_country"].replace("United States", "USA", inplace=True)
df["first_country"].replace("United Kingdom", "UK", inplace=True)
df["first_country"].replace("South Korea", "S. Korea", inplace=True)

In [ ]:
### cell 9 ###

data = df.groupby("first_country")["count"].sum().sort_values(ascending=False)[:10]

In [ ]:
### cell 10 ###

country_order = df["first_country"].value_counts()[:11].index
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df = df._to_pandas()

data_q2q3 = (
    df[["type", "first_country"]]
    .groupby("first_country")["type"]
    .value_counts()
    .unstack()
    .loc[country_order]
)
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df = pd.DataFrame(df)
    data_q2q3 = pd.DataFrame(data_q2q3)

data_q2q3["sum"] = data_q2q3.sum(axis=1)
data_q2q3_ratio = (
    (data_q2q3.T / data_q2q3["sum"])
    .T[["Movie", "TV Show"]]
    .sort_values(by="Movie", ascending=False)[::-1]
)

In [ ]:
### cell 11 ###

# Skip sorting the group keys to avoid an extra sort step
rating_order = (
    df.groupby("rating", sort=False)["count"]
    .sum()
    .sort_values(ascending=False)
    .index.tolist()
)

In [ ]:
### cell 12 ###

use_modin = os.environ.get("IREWR_WITH_MODIN") == "True"
# work on a pandas DataFrame if using Modin, leave df untouched
df_input = df._to_pandas() if use_modin else df
# compute the contingency table in one pass and reorder columns
mf = pd.crosstab(df_input["type"], df_input["rating"]).loc[:, rating_order].sort_index()
# convert the result back to Modin if needed
if use_modin:
    mf = pd.DataFrame(mf)
# extract the two series
movie = mf.loc["Movie"]
tv = -mf.loc["TV Show"]

In [ ]:
### cell 13 ###

modin_flag = os.environ.get("IREWR_WITH_MODIN") == "True"
# Use a pandas view of df if running under Modin, otherwise operate on df directly
df_p = df._to_pandas() if modin_flag else df
# Compute counts with crosstab (faster than groupby->value_counts->unstack), select types, cumsum over types, then transpose
data_sub = (
    pd.crosstab(df_p["type"], df_p["year_added"])
    .loc[["TV Show", "Movie"]]
    .cumsum(axis=0)
    .T
)
# If Modin was in use, wrap the result back into a Modin DataFrame
if modin_flag:
    data_sub = pd.DataFrame(data_sub)

In [ ]:
### cell 14 ###

month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]

df["index"] = pd.Categorical(df["index"], categories=month_order, ordered=True)

In [ ]:
### cell 15 ###

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df = df._to_pandas()

data_sub = (
    df.groupby("type")["index"]
    .value_counts()
    .unstack()
    .fillna(0)
    .loc[["TV Show", "Movie"]]
    .cumsum(axis=0)
    .T
)
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df = pd.DataFrame(df)
    data_sub = pd.DataFrame(data_sub)

In [ ]:
### cell 16 ###

data_sub2 = data_sub

data_sub2["Value"] = data_sub2["Movie"] + data_sub2["TV Show"]
data_sub2 = data_sub2.reset_index()

df_polar = data_sub2.sort_values(by="index", ascending=False)


color_map = ["#221f1f" for _ in range(12)]
color_map[0] = color_map[11] = "#b20710"  # color highlight

# Constants = parameters controling the plot layout:
upperLimit = 30
lowerLimit = 1
labelPadding = 30

# Compute max and min in the dataset
max = df_polar["Value"].max()

# Let's compute heights: they are a conversion of each item value in those new coordinates
# In our example, 0 in the dataset will be converted to the lowerLimit (10)
# The maximum will be converted to the upperLimit (100)
slope = (max - lowerLimit) / max
heights = slope * df_polar.Value + lowerLimit

# Compute the width of each bar. In total we have 2*Pi = 360°
width = 2 * np.pi / len(df_polar.index)

# Compute the angle each bar is centered on:
indexes = list(range(1, len(df_polar.index) + 1))
angles = [element * width for element in indexes]
angles

In [ ]:
### cell 17 ###

# Genres
from sklearn.preprocessing import MultiLabelBinarizer

import matplotlib.colors


# Custom colour map based on Netflix palette
cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
    "", ["#221f1f", "#b20710", "#f5f5f1"]
)

In [ ]:
### cell 18 ###


def genre_heatmap(df, title):
    df["genre"] = df["listed_in"].apply(
        lambda x: x.replace(" ,", ",").replace(", ", ",").split(",")
    )
    Types = []
    for i_3 in df["genre"]:
        Types += i_3
    Types = set(Types)
    print("There are {} types in the Netflix {} Dataset".format(len(Types), title))
    test_1 = df["genre"]


df_tv = df[df["type"] == "TV Show"]
df_movies = df[df["type"] == "Movie"]


genre_heatmap(df_movies, "Movie")

In [ ]:
### cell 19 ###

# Compute top 10 countries by total count without copying the index into a Series
data = df["count"].groupby(df["first_country"]).sum().nlargest(10).index

# Filter using the boolean mask via .loc
# (using the index directly avoids an extra small Series allocation)
df_heatmap = df.loc[df["first_country"].isin(data)]

In [ ]:
### cell 20 ###

df_heatmap = pd.crosstab(
    df_heatmap["first_country"], df_heatmap["target_ages"], normalize="index"
).T

In [ ]:
### cell 21 ###

# Data

df_movies
df_tv

### Relevant groupings

data = (
    df_movies.groupby("first_country")[["count"]]
    .sum()
    .sort_values(by="count", ascending=False)
    .reset_index()[:10]
)
data = data["first_country"]
df_loli = df_movies.loc[df_movies["first_country"].isin(data)]

loli = df_loli.groupby("first_country")[["release_year", "year_added"]].mean().round()


# Reorder it following the values of the first value
ordered_df = loli.sort_values(by="release_year")

ordered_df_rev = loli.sort_values(by="release_year", ascending=False)

In [ ]:
### cell 22 ###

# Aggregate count and compute mean years in a single pass
stats = df_tv.groupby("first_country", sort=False).agg(
    count=("count", "sum"),
    release_year=("release_year", "mean"),
    year_added=("year_added", "mean"),
)
# Round the year columns
stats[["release_year", "year_added"]] = stats[["release_year", "year_added"]].round()
# Select top 10 countries by total count
data = stats["count"].nlargest(10).index
# Filter original DataFrame (preserves df_loli for downstream use)
df_loli = df_tv[df_tv["first_country"].isin(data)]
# Extract the averaged-year metrics for those top countries
loli = stats.loc[data, ["release_year", "year_added"]]
# Sort ascending and descending by release_year
ordered_df = loli.sort_values(by="release_year")
ordered_df_rev = loli.sort_values(by="release_year", ascending=False)

In [ ]:
### cell 23 ###

use_modin = os.environ.get("IREWR_WITH_MODIN") == "True"
countries = ["USA", "India"]

# 1) Fast boolean filter for us_ind (on original df, whether modin or pandas)
mask = df["first_country"].isin(countries)
us_ind = df.loc[mask]

# 2) If using Modin, convert just for the heavy groupby; leave df itself untouched
df_p = df._to_pandas() if use_modin else df

# 3) Compute the same cumulative cross‐country sums by pivoting on year first,
#    then cumsum across the country columns (axis=1) to match original .cumsum(axis=0).T
data_sub = (
    df_p.loc[df_p["first_country"].isin(countries), ["year_added", "first_country"]]
    .groupby(["year_added", "first_country"])  # group on year, then country
    .size()  # fast count
    .unstack(fill_value=0)  # pivot into year×country
    .loc[:, countries]  # ensure column order [USA, India]
    .cumsum(axis=1)  # accumulate across country columns
    .astype(float)  # match original float dtype
)

In [ ]:
### cell 24 ###

# Filter for USA and India once
us_ind = df[df["first_country"].isin(["USA", "India"])]

# Convert Modin to pandas if needed
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df = df._to_pandas()

# Compute raw counts by year and country in one groupby, only for the two countries
counts = (
    df[df["first_country"].isin(["USA", "India"])]
    .groupby(["year_added", "first_country"])
    .size()
    .unstack(fill_value=0)
)

# Combined counts and USA-only counts
combined = counts.sum(axis=1)
us_counts = counts["USA"]
half_comb = combined / 2

# Build the final DataFrame with base, USA, India columns
data_sub = pd.DataFrame(
    {"base": -half_comb, "USA": us_counts - half_comb, "India": half_comb}
)

# Convert back to Modin if needed
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df = pd.DataFrame(df)
    data_sub = pd.DataFrame(data_sub)

In [ ]:
### cell 25 ###

# Optimized single-step concatenation and cleanup
text = (
    df["title"]
    .astype(str)
    .str.cat(sep=" ")
    .replace(",", "")
    .replace("'", "")
    .replace(".", "")
)